# 03 Appendix Text Figures

Produces figures for the appendix of the manuscript. Reads fitted `.pkl` artifacts from `01_analysis.ipynb`, `02_simulation.ipynb`, and related figure notebooks. No model refitting happens here.

Appendix figure sections should be added below the shared setup cells as each figure is specified.

**Figure list:**

1. **Appendix Figure 1**: Per conspiracy adoption prevalence, empirical versus baseline simulation.
2. **Appendix Figure 2**: First adoption entry point scatter, empirical versus baseline simulation.
3. **Appendix Figure 3**: Baseline diffusion curves.
4. **Appendix Figure 4**: Counterfactual scenarios ranked by total AUC.
5. **Appendix Figure 5**: Hawkes model comparison and simulation check.
6. **Appendix Figure 6**: Empirical and simulated coadoption Jaccard matrices.
7. **Appendix Figure 7**: Exposure hazard ratio comparison across sequential adoption models.

In [ ]:
# Imports and publication style
%matplotlib inline
import sys, os, joblib
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from conspiracy_analysis.visualization import (
    set_nhb_style,
    NHB_COL_SINGLE, NHB_COL_ONE_HALF, NHB_COL_DOUBLE,
    NHB_PALETTE, CLUSTER_PALETTE,
    apply_panel_label, nhb_annotation_fontsize,
)
from conspiracy_analysis.visualization.plots import _CONSPIRACY_LABELS
from conspiracy_analysis.simulation.provenance import load_manifest_backed_simulation_bundle

def display_conspiracy_label(name):
    short = name.replace('ConsProb_', '')
    return _CONSPIRACY_LABELS.get(short, short)

set_nhb_style()

# Output directories for paper and optional figure renders.
FIGURE_FINAL_DIR = os.path.join('..', 'figures_final')
FIGURE_OPTIONAL_DIR = os.path.join('..', 'figures_optional')
os.makedirs(FIGURE_FINAL_DIR, exist_ok=True)
os.makedirs(FIGURE_OPTIONAL_DIR, exist_ok=True)
print(f'Paper figures will be saved to: {os.path.abspath(FIGURE_FINAL_DIR)}')
print(f'Optional figures will be saved to: {os.path.abspath(FIGURE_OPTIONAL_DIR)}')

SIM_BOOTSTRAP_N = 2000
SIM_BOOTSTRAP_SEED = 20240517

def bootstrap_mean_ci(values, n_boot=SIM_BOOTSTRAP_N, seed=SIM_BOOTSTRAP_SEED):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    estimate = float(np.mean(values)) if values.size else float('nan')
    if values.size == 0:
        return estimate, float('nan'), float('nan')
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, values.size, size=(n_boot, values.size))
    boot = values[idx].mean(axis=1)
    lower, upper = np.percentile(boot, [2.5, 97.5])
    return estimate, float(lower), float(upper)

def bootstrap_auc_change_ci(
    baseline_values,
    scenario_values,
    intensity=None,
    n_boot=SIM_BOOTSTRAP_N,
    seed=SIM_BOOTSTRAP_SEED,
):
    baseline_values = np.asarray(baseline_values, dtype=float)
    scenario_values = np.asarray(scenario_values, dtype=float)
    if baseline_values.shape != scenario_values.shape:
        raise ValueError('Baseline and scenario arrays must have matching shapes.')
    if baseline_values.size == 0:
        return {
            'change': float('nan'),
            'change_ci': (float('nan'), float('nan')),
            'efficiency': float('nan'),
            'efficiency_ci': (float('nan'), float('nan')),
        }
    baseline_mean = float(np.mean(baseline_values))
    scenario_mean = float(np.mean(scenario_values))
    change = (scenario_mean / baseline_mean - 1.0) * 100.0
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, baseline_values.size, size=(n_boot, baseline_values.size))
    boot_baseline = baseline_values[idx].mean(axis=1)
    boot_scenario = scenario_values[idx].mean(axis=1)
    boot_change = (boot_scenario / boot_baseline - 1.0) * 100.0
    change_lower, change_upper = np.percentile(boot_change, [2.5, 97.5])
    result = {
        'change': change,
        'change_ci': (float(change_lower), float(change_upper)),
    }
    if intensity is not None:
        efficiency = -change / intensity
        boot_efficiency = -boot_change / intensity
        eff_lower, eff_upper = np.percentile(boot_efficiency, [2.5, 97.5])
        result.update({
            'efficiency': efficiency,
            'efficiency_ci': (float(eff_lower), float(eff_upper)),
        })
    return result

def ci_error_array(values, lowers, uppers):
    values = np.asarray(values, dtype=float)
    lowers = np.asarray(lowers, dtype=float)
    uppers = np.asarray(uppers, dtype=float)
    invalid = (lowers > values) | (uppers < values)
    if np.any(invalid):
        raise ValueError('CI bounds must bracket point estimates.')
    return np.vstack([
        values - lowers,
        uppers - values,
    ])

In [ ]:
from pathlib import Path

INTERMEDIATE_DIR = Path("../intermediate_files")
INTERMEDIATE_DIR.mkdir(exist_ok=True)

def intermediate_path(filename):
    return INTERMEDIATE_DIR / filename


In [ ]:
# Load fitted Cox models from pkl files.
# Per conspiracy coefficient models.
model_names = [
    'model_1', 'model_2a', 'model_2b',
    'model_3', 'model_4', 'model_5',
    'model_6', 'model_7', 'model_8',
]
cox_results = {}
for name in model_names:
    pkl_path = intermediate_path(f'{name}_cox.pkl')
    if os.path.exists(pkl_path):
        ctv = joblib.load(pkl_path)
        cox_results[name] = (ctv, None)
        print(f'Loaded {pkl_path}: {len(ctv.params_)} parameters')
    else:
        print(f'WARNING: {pkl_path} not found. Run 01_analysis.ipynb first')

# Stratified variants that appendix figures can reuse.
stratified_models = {}
for name in ['model_1_strat', 'model_pooled']:
    pkl_path = intermediate_path(f'{name}_cox.pkl')
    if os.path.exists(pkl_path):
        stratified_models[name] = joblib.load(pkl_path)
        print(f'Loaded {pkl_path}: {len(stratified_models[name].params_)} parameters')
    else:
        print(f'WARNING: {pkl_path} not found. Run 01_analysis.ipynb first')

# Parametrize baseline hazards for appendix figures.
from conspiracy_analysis.models.baseline_hazards import (
    parametrize_all_baselines, compute_all_decay_times,
)
from conspiracy_analysis.models.bootstrap_inference import load_bootstrap_artifact
cox_models_dict = {n: ctv for n, (ctv, _) in cox_results.items() if n != 'model_2b'}
all_baselines = parametrize_all_baselines(cox_models_dict)
decay_times = compute_all_decay_times(all_baselines)
print(f'\nParametrized {len(all_baselines)} baseline hazards')
cox_bootstrap_results = load_bootstrap_artifact(intermediate_path('cox_bootstrap_results.pkl'))
timeline_bootstrap_results = load_bootstrap_artifact(intermediate_path('timeline_bootstrap_results.pkl'))
print('Loaded bootstrap interval artifacts')


In [ ]:
# Load graph, semantic matrix, clustering, barrier structures, and settler structures.
import networkx as nx
from conspiracy_analysis import BOT_SCORE_THRESHOLD
from conspiracy_analysis.data.loader import load_graph_and_tweets
from conspiracy_analysis.config import (
    apply_conspiracy_config_to_graph,
    get_baseline_conspiracy,
    get_included_conspiracies,
)
from conspiracy_analysis.analysis.semantic import (
    find_optimal_clustering, assign_clusters, load_clustering_result,
    compute_temporal_distance_matrix, compute_peak_frequency_distance_matrix,
)
from conspiracy_analysis.analysis.statistics import (
    extract_user_timelines, compute_semantic_barrier_analysis, compute_settler_effect,
)

G, _ = load_graph_and_tweets(from_joblib=True, time_resolution='Hour')
G = G.to_undirected()
G.remove_nodes_from(list(nx.isolates(G)))

# Reference and analytic categories come from config/conspiracies.yaml.
BASELINE_CONSPIRACY = get_baseline_conspiracy()
INCLUDED_CONSPIRACIES = get_included_conspiracies()
G = apply_conspiracy_config_to_graph(G)
conspiracies = G.graph['conspiracy_cols']

# Semantic clustering saved by 01_analysis.ipynb.
clustering_result = load_clustering_result()
cluster_assignments = assign_clusters(clustering_result)

# Temporal first appearance clustering.
df_temporal = compute_temporal_distance_matrix(G)
temporal_clustering = find_optimal_clustering(
    df_temporal, conspiracies,
    methods=['ward', 'average', 'complete', 'single'], k_range=range(3, 9),
)
temporal_cluster_assignments = assign_clusters(temporal_clustering)

# Peak frequency clustering.
df_peak_freq = compute_peak_frequency_distance_matrix(G, window=24)
peak_freq_clustering = find_optimal_clustering(
    df_peak_freq, conspiracies,
    methods=['ward', 'average', 'complete', 'single'], k_range=range(3, 9),
)
peak_freq_cluster_assignments = assign_clusters(peak_freq_clustering)

# Semantic barrier and settler measures.
user_timelines = extract_user_timelines(
    G, cluster_assignments, bot_score_threshold=BOT_SCORE_THRESHOLD,
    mode='HUMAN', min_conspiracies=2,
)
semantic_gaps = compute_semantic_barrier_analysis(user_timelines, cluster_assignments)
semantic_settler = compute_settler_effect(user_timelines, cluster_assignments)

# Temporal first appearance barrier and settler measures.
temporal_timelines = extract_user_timelines(
    G, temporal_cluster_assignments, bot_score_threshold=BOT_SCORE_THRESHOLD,
    mode='HUMAN', min_conspiracies=2,
)
temporal_gaps = compute_semantic_barrier_analysis(temporal_timelines, temporal_cluster_assignments)
temporal_settler = compute_settler_effect(temporal_timelines, temporal_cluster_assignments)

# Peak frequency barrier and settler measures.
peak_freq_timelines = extract_user_timelines(
    G, peak_freq_cluster_assignments, bot_score_threshold=BOT_SCORE_THRESHOLD,
    mode='HUMAN', min_conspiracies=2,
)
peak_freq_gaps = compute_semantic_barrier_analysis(peak_freq_timelines, peak_freq_cluster_assignments)
peak_freq_settler = compute_settler_effect(peak_freq_timelines, peak_freq_cluster_assignments)

print(f'\nIncluded conspiracies: {len(conspiracies)}')
print(f'Semantic clustering: {clustering_result["best_method"]} linkage, k={clustering_result["best_k"]}, '
      f'silhouette={clustering_result["best_score"]:.4f}')
print(f'Temporal first appearance clustering: {temporal_clustering["best_method"]} linkage, '
      f'k={temporal_clustering["best_k"]}, silhouette={temporal_clustering["best_score"]:.4f}')
print(f'Peak frequency clustering: {peak_freq_clustering["best_method"]} linkage, '
      f'k={peak_freq_clustering["best_k"]}, silhouette={peak_freq_clustering["best_score"]:.4f}')


In [ ]:
# Load the accepted aggregate once for every simulation based appendix figure.
bundle = load_manifest_backed_simulation_bundle(
    intermediate_path('simulation_results_all.pkl')
)
results_baseline = bundle['results_baseline']
cfg = bundle['config']
counterfactual_steps = cfg['counterfactual_steps']
print('Loaded manifest backed simulation_results_all.pkl')

## Appendix Figure Sections

Add each requested appendix figure below this cell.

## Appendix Figure 1

Per conspiracy adoption prevalence, empirical versus baseline simulation.

In [ ]:
from scipy.stats import pearsonr
import networkx as nx
from conspiracy_analysis import BOT_SCORE_THRESHOLD
from conspiracy_analysis.config import apply_conspiracy_config_to_graph
from conspiracy_analysis.utils.helpers import passes_bot_filter
from conspiracy_analysis.visualization import NHB_COL_DOUBLE, NHB_PALETTE, apply_panel_label

if 'empirical_adoption_share' not in globals():
    G_prevalence_empirical, _ = load_graph_and_tweets(from_joblib=True, time_resolution='Hour')
    print('Loaded G_MC.pkl')
    nodes_to_remove = [
        node for node, data in G_prevalence_empirical.nodes(data=True)
        if not passes_bot_filter(G_prevalence_empirical, node, BOT_SCORE_THRESHOLD, 'HUMAN')
    ]
    G_prevalence_empirical.remove_nodes_from(nodes_to_remove)
    giant_nodes = max(nx.connected_components(G_prevalence_empirical.to_undirected()), key=len)
    G_prevalence_empirical = G_prevalence_empirical.subgraph(giant_nodes).copy()
    G_prevalence_empirical = apply_conspiracy_config_to_graph(G_prevalence_empirical)
    n_total_empirical = G_prevalence_empirical.number_of_nodes()
    empirical_adoption_share = {}
    for c in G_prevalence_empirical.graph['conspiracy_cols']:
        n_adopters = 0
        for node in G_prevalence_empirical.nodes():
            acts = G_prevalence_empirical.nodes[node].get(c, [])
            if acts and not all(np.isnan(a) for a in acts):
                n_adopters += 1
        empirical_adoption_share[c] = n_adopters / n_total_empirical

conspiracies_list = sorted(empirical_adoption_share.keys())
emp_shares_vals = [empirical_adoption_share[c] for c in conspiracies_list]

sim_shares_per_run = []
for run in results_baseline.runs:
    counts = {c: 0 for c in conspiracies_list}
    for node_hist in run.adoption_histories.values():
        for c in node_hist:
            if c in counts:
                counts[c] += 1
    sim_shares_per_run.append({c: counts[c] / run.n_nodes for c in conspiracies_list})

sim_means = []
sim_ci_lowers = []
sim_ci_uppers = []
for c in conspiracies_list:
    mean, lower, upper = bootstrap_mean_ci([r[c] for r in sim_shares_per_run])
    sim_means.append(mean)
    sim_ci_lowers.append(lower)
    sim_ci_uppers.append(upper)

order = np.argsort(emp_shares_vals)[::-1]
conspiracies_sorted = [conspiracies_list[i] for i in order]
emp_sorted = [emp_shares_vals[i] for i in order]
sim_sorted = [sim_means[i] for i in order]
sim_lower_sorted = [sim_ci_lowers[i] for i in order]
sim_upper_sorted = [sim_ci_uppers[i] for i in order]
short_labels = [display_conspiracy_label(c) for c in conspiracies_sorted]
sim_yerr = ci_error_array(sim_sorted, sim_lower_sorted, sim_upper_sorted)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=NHB_COL_DOUBLE)
x = np.arange(len(short_labels))
ax1.bar(x - 0.2, emp_sorted, 0.4, label='Empirical', color=NHB_PALETTE['positive'], linewidth=0)
ax1.bar(x + 0.2, sim_sorted, 0.4, yerr=sim_yerr, label='Simulated baseline',
        color=NHB_PALETTE['highlight'], capsize=3, linewidth=0)
ax1.set_xticks(x)
ax1.set_xticklabels(short_labels, rotation=45, ha='right', rotation_mode='anchor')
ax1.set_ylabel('Share of users who adopted')
ax1.set_title('Per Conspiracy Adoption Prevalence')
ax1.legend()
apply_panel_label(ax1, 'a')

r_val, _ = pearsonr(emp_sorted, sim_sorted)
emp_arr = np.asarray(emp_sorted, dtype=float)
sim_arr = np.asarray(sim_sorted, dtype=float)
ss_res = np.sum((emp_arr - sim_arr) ** 2)
ss_tot = np.sum((emp_arr - emp_arr.mean()) ** 2)
r2_val = 1 - ss_res / ss_tot if ss_tot > 0 else float('nan')
marker_cycle = ['o', 's', '^', 'D', 'P', 'X', 'v', '<', '>', '*', 'h', 'H', 'p', '8', 'd']
color_indices = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19]
point_cmap = plt.get_cmap('tab20')
for i, (label, emp_val, sim_val) in enumerate(zip(short_labels, emp_sorted, sim_sorted)):
    ax2.scatter(emp_val, sim_val, s=85, alpha=0.9,
                color=point_cmap(color_indices[i % len(color_indices)]),
                marker=marker_cycle[i % len(marker_cycle)],
                edgecolor='white', linewidth=0.6, label=label, zorder=3)
lims = [0, max(max(emp_sorted), max(sim_sorted)) * 1.1]
ax2.plot(lims, lims, '--', color='grey', alpha=0.5, linewidth=0.5)
ax2.set_xlim(lims)
ax2.set_ylim(lims)
ax2.set_xlabel('Empirical adoption share')
ax2.set_ylabel('Simulated adoption share')
ax2.set_title('Per Conspiracy Comparison')
ax2.legend(title='Conspiracy', loc='upper left', bbox_to_anchor=(1.02, 1),
           ncol=1, frameon=False, borderaxespad=0, handletextpad=0.4,
           labelspacing=0.3, scatterpoints=1)
apply_panel_label(ax2, 'b')

print(f'r = {r_val:.3f}, R^2 = {r2_val:.3f}')
plt.tight_layout()
save_path = os.path.join(FIGURE_FINAL_DIR, 'figA01_per_conspiracy_prevalence.png')
fig.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

## Appendix Figure 2

First adoption entry point scatter, empirical versus baseline simulation.

In [ ]:
from conspiracy_analysis.simulation.evaluation import (
    compute_first_adoption_distribution,
    compute_empirical_comparison,
)
import networkx as nx
from conspiracy_analysis import BOT_SCORE_THRESHOLD
from conspiracy_analysis.config import apply_conspiracy_config_to_graph
from conspiracy_analysis.utils.helpers import passes_bot_filter
from conspiracy_analysis.visualization import NHB_COL_ONE_HALF, NHB_PALETTE
from conspiracy_analysis.visualization import apply_panel_label

if 'empirical_portions' not in globals():
    G_first_adoption_empirical, _ = load_graph_and_tweets(from_joblib=True, time_resolution='Hour')
    print('Loaded G_MC.pkl')
    nodes_to_remove = [
        node for node, data in G_first_adoption_empirical.nodes(data=True)
        if not passes_bot_filter(G_first_adoption_empirical, node, BOT_SCORE_THRESHOLD, 'HUMAN')
    ]
    G_first_adoption_empirical.remove_nodes_from(nodes_to_remove)
    giant_nodes = max(nx.connected_components(G_first_adoption_empirical.to_undirected()), key=len)
    G_first_adoption_empirical = G_first_adoption_empirical.subgraph(giant_nodes).copy()
    G_first_adoption_empirical = apply_conspiracy_config_to_graph(G_first_adoption_empirical)
    first_adoption_counts = {c: 0 for c in G_first_adoption_empirical.graph['conspiracy_cols']}
    total_active = 0
    for node in G_first_adoption_empirical.nodes():
        first_times = {}
        for c in G_first_adoption_empirical.graph['conspiracy_cols']:
            activations = G_first_adoption_empirical.nodes[node].get(c, [])
            if activations:
                first_times[c] = min(activations)
        if first_times:
            first_conspiracy = min(first_times, key=first_times.get)
            first_adoption_counts[first_conspiracy] += 1
            total_active += 1
    empirical_portions = {
        c: first_adoption_counts[c] / total_active
        for c in G_first_adoption_empirical.graph['conspiracy_cols']
    }

summary_df, _ = compute_first_adoption_distribution(results_baseline)
metrics = compute_empirical_comparison(results_baseline, empirical_portions)

fig, ax = plt.subplots(figsize=NHB_COL_ONE_HALF)
shared = set(summary_df['conspiracy']) & set(empirical_portions.keys())
marker_cycle = ['o', 's', '^', 'D', 'P', 'X', 'v', '<', '>', '*', 'h', 'H', 'p', '8', 'd']
color_indices = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19]
point_cmap = plt.get_cmap('tab20')
for i, c in enumerate(sorted(shared, key=lambda item: -empirical_portions[item])):
    row = summary_df[summary_df['conspiracy'] == c]
    if not row.empty:
        sim_val = row['mean_frequency'].iloc[0]
        emp_val = empirical_portions[c]
        label = display_conspiracy_label(c)
        ax.scatter(emp_val, sim_val, s=85, alpha=0.9,
                   color=point_cmap(color_indices[i % len(color_indices)]),
                   marker=marker_cycle[i % len(marker_cycle)],
                   edgecolor='white', linewidth=0.6, label=label, zorder=3)

lims = [0, 0.4]
ax.plot(lims, lims, '--', color='grey', alpha=0.5, linewidth=0.5)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('Empirical proportion')
ax.set_ylabel('Simulated proportion')
ax.set_title('First Adoption Distribution')
ax.legend(title='Conspiracy', loc='upper left', bbox_to_anchor=(1.02, 1),
          ncol=1, frameon=False, borderaxespad=0, handletextpad=0.4,
          labelspacing=0.3, scatterpoints=1)
print(f"R^2 = {metrics['r_squared']:.3f}")
plt.tight_layout()
save_path = os.path.join(FIGURE_FINAL_DIR, 'figA02_first_adoption_scatter.png')
fig.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

## Appendix Figure 3

Baseline diffusion curves.

In [ ]:
import pandas as pd
from conspiracy_analysis.simulation.evaluation import compute_diffusion_curves
from conspiracy_analysis.visualization import NHB_COL_DOUBLE, NHB_PALETTE
from conspiracy_analysis.analysis.semantic import build_cluster_display_metadata

df_baseline = compute_diffusion_curves(results_baseline, steps=counterfactual_steps)
baseline_records = []
for (consp, t), group in df_baseline.groupby(['conspiracy', 't'], sort=False):
    mean, lower, upper = bootstrap_mean_ci(group['adoption_fraction'].values)
    baseline_records.append({
        'conspiracy': consp,
        't': t,
        'mean': mean,
        'ci_lower': lower,
        'ci_upper': upper,
    })
mean_baseline = pd.DataFrame(baseline_records)

cluster_label_lookup, cluster_color_lookup = build_cluster_display_metadata(
    cluster_assignments
)
linestyle_cycle = ['-', '--', ':', '-.']

fig, ax = plt.subplots(figsize=NHB_COL_DOUBLE)
within_cluster_idx = {}
for consp in sorted(mean_baseline['conspiracy'].unique()):
    short = consp.replace('ConsProb_', '')
    cluster_name = cluster_label_lookup.get(short, 'Unknown')
    color = cluster_color_lookup.get(cluster_name, NHB_PALETTE['neutral'])
    k = within_cluster_idx.get(cluster_name, 0)
    within_cluster_idx[cluster_name] = k + 1
    style = linestyle_cycle[k % len(linestyle_cycle)]

    sub = mean_baseline[mean_baseline['conspiracy'] == consp].sort_values('t')
    ax.plot(sub['t'], sub['mean'], label=display_conspiracy_label(consp),
            color=color, linestyle=style, linewidth=2.5)
    ax.fill_between(sub['t'], sub['ci_lower'], sub['ci_upper'],
                    color=NHB_PALETTE['neutral'], alpha=0.08, linewidth=0)

ax.set_xlabel('Time step, hours')
ax.set_ylabel('Adoption fraction')
ax.set_title('Baseline Adoption Curves Over Time', fontsize=18)
ax.legend(loc='upper left', ncol=2, frameon=False, fontsize=14)

plt.tight_layout()
save_path = os.path.join(FIGURE_FINAL_DIR, 'figA03_baseline_diffusion_curves.png')
fig.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')


## Appendix Figure 4

Counterfactual scenarios ranked by total AUC.

In [ ]:
import pandas as pd
from conspiracy_analysis.simulation.evaluation import compute_diffusion_curves
from conspiracy_analysis.visualization import NHB_COL_DOUBLE, NHB_PALETTE, CLUSTER_PALETTE
from conspiracy_analysis.visualization import nhb_annotation_fontsize, apply_panel_label

def _total_auc_by_run(results):
    df = compute_diffusion_curves(results, steps=counterfactual_steps)
    auc_records = []
    for (run, consp), group in df.groupby(['RUN', 'conspiracy']):
        group = group.sort_values('t')
        auc = np.trapezoid(group['adoption_fraction'].values, group['t'].values)
        auc_records.append({'RUN': run, 'auc': auc})
    df_auc = pd.DataFrame(auc_records)
    return df_auc.groupby('RUN')['auc'].sum().sort_index()

baseline_auc_by_run = _total_auc_by_run(bundle['results_baseline'])
baseline_auc_mean = float(baseline_auc_by_run.mean())

all_scenarios = {
    'baseline': bundle['results_baseline'],
    'quarantine_4h': bundle['results_quarantine_4h'],
    'quarantine_12h': bundle['results_quarantine_12h'],
    'quarantine_24h': bundle['results_quarantine_24h'],
    'quarantine_168h': bundle['results_quarantine_168h'],
    'no_temporal_effects': bundle['results_no_temporal'],
}
for r in sorted(bundle['moderation_results'].keys()):
    all_scenarios[f'moderation_{int(r * 100)}pct'] = bundle['moderation_results'][r]
for r in sorted(bundle['nudge_results'].keys()):
    all_scenarios[f'nudge_{int(r * 100)}pct'] = bundle['nudge_results'][r]

rows = []
for name, results in all_scenarios.items():
    total_per_run = _total_auc_by_run(results)
    m, lower, upper = bootstrap_mean_ci(total_per_run.values)
    if name == 'baseline':
        pct = 0.0
    else:
        common_runs = baseline_auc_by_run.index.intersection(total_per_run.index)
        if len(common_runs) != len(baseline_auc_by_run) or len(common_runs) != len(total_per_run):
            raise ValueError(f'Baseline and {name} run ids must match for paired bootstrap.')
        change_summary = bootstrap_auc_change_ci(
            baseline_auc_by_run.loc[common_runs].values,
            total_per_run.loc[common_runs].values,
        )
        pct = -change_summary['change']
    rows.append({
        'scenario': name,
        'total_auc_mean': m,
        'ci_lower': lower,
        'ci_upper': upper,
        'pct_reduction_vs_baseline': pct,
    })

df_auc_summary = pd.DataFrame(rows)
df_plot = df_auc_summary.sort_values('total_auc_mean', ascending=True)

family_colors = {
    'baseline': NHB_PALETTE['positive'],
    'moderation': CLUSTER_PALETTE['Political'],
    'nudge': CLUSTER_PALETTE['Pandemic'],
    'quarantine': NHB_PALETTE['highlight'],
    'no_temporal_effects': CLUSTER_PALETTE['Biomedical'],
}

def _scenario_family(name):
    if name == 'baseline':
        return 'baseline'
    for prefix in ('moderation', 'nudge', 'quarantine'):
        if name.startswith(prefix):
            return prefix
    return 'no_temporal_effects'

colors = [family_colors[_scenario_family(n)] for n in df_plot['scenario']]

fig_height = max(NHB_COL_DOUBLE[1], len(df_plot) * 0.18)
fig, ax = plt.subplots(figsize=(NHB_COL_DOUBLE[0], fig_height))
xerr = ci_error_array(
    df_plot['total_auc_mean'].values,
    df_plot['ci_lower'].values,
    df_plot['ci_upper'].values,
)
bars = ax.barh(
    range(len(df_plot)),
    df_plot['total_auc_mean'].values,
    xerr=xerr,
    color=colors,
    capsize=3,
    edgecolor='white',
    linewidth=0,
)

x_max = df_plot['total_auc_mean'].max() * 1.45
label_x = x_max * 0.98
for i, (_, row) in enumerate(df_plot.iterrows()):
    label = f'{row["total_auc_mean"]:.1f}'
    if row['scenario'] != 'baseline':
        label += f'  ({row["pct_reduction_vs_baseline"]:.1f}% reduction)'
    ax.text(
        label_x,
        bars[i].get_y() + bars[i].get_height() / 2,
        label,
        va='center',
        ha='right',
        fontsize=nhb_annotation_fontsize(),
    )

ax.set_yticks(range(len(df_plot)))
ax.set_yticklabels(df_plot['scenario'].values)
ax.set_xlabel('Total AUC, adoption fraction x hours')
ax.set_title('All Counterfactuals: Total Adoption AUC')
ax.set_xlim(0, x_max)
plt.tight_layout()
save_path = os.path.join(FIGURE_FINAL_DIR, 'figA04_auc_total_all_scenarios.png')
fig.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

## Appendix Figure 5

Hawkes model comparison and simulation check.

In [ ]:
import networkx as nx
from conspiracy_analysis import BOT_SCORE_THRESHOLD
from conspiracy_analysis.config import apply_conspiracy_config_to_graph, get_included_conspiracies
from conspiracy_analysis.models.hawkes_sharing import (
    HawkesFitResult,
    extract_sharing_sequences,
    hawkes_model_comparison,
)
from conspiracy_analysis.utils.helpers import passes_bot_filter
from conspiracy_analysis.visualization import plot_hawkes_goodness_of_fit

hawkes_config = bundle['config']
hawkes_fit = HawkesFitResult(**hawkes_config['hawkes_fit'])

G_hawkes, _ = load_graph_and_tweets(from_joblib=True, time_resolution='Hour')
nodes_to_remove = [
    node for node, data in G_hawkes.nodes(data=True)
    if not passes_bot_filter(G_hawkes, node, BOT_SCORE_THRESHOLD, 'HUMAN')
]
G_hawkes.remove_nodes_from(nodes_to_remove)
giant_nodes = max(nx.connected_components(G_hawkes.to_undirected()), key=len)
G_hawkes = G_hawkes.subgraph(giant_nodes).copy()
G_hawkes = apply_conspiracy_config_to_graph(G_hawkes)
hawkes_conspiracies = hawkes_config.get('CONSPIRACIES')
if hawkes_conspiracies is None:
    hawkes_conspiracies = get_included_conspiracies()

study_end = max(
    t
    for node in G_hawkes.nodes()
    for consp in hawkes_conspiracies
    for t in G_hawkes.nodes[node].get(consp, [])
)
sequences, observation_ends = extract_sharing_sequences(
    G_hawkes,
    hawkes_conspiracies,
    study_end=study_end,
)

save_path = os.path.join(FIGURE_FINAL_DIR, 'figA05_hawkes_goodness_of_fit.png')
fig = plot_hawkes_goodness_of_fit(
    sequences,
    hawkes_fit,
    observation_ends=observation_ends,
    save_path=save_path,
)
plt.show()

print('=== Model Comparison (AIC / BIC) ===')
comp = hawkes_model_comparison(sequences, hawkes_fit, observation_ends=observation_ends)
print(comp.to_string(index=False))
print(f"\n  Best model by AIC: {comp.loc[comp['AIC'].idxmin(), 'model']}")
delta_aic = (
    comp.loc[comp['model'] == 'Homogeneous Poisson', 'AIC'].values[0]
    - comp.loc[comp['model'] == 'Hawkes', 'AIC'].values[0]
)
print(f'  Delta AIC, Hawkes versus Poisson: {delta_aic:.0f}')
print(f'Saved: {save_path}')

## Appendix Figure 6

Empirical and simulated coadoption Jaccard matrices.

In [ ]:
import seaborn as sns
import pandas as pd
import networkx as nx
from conspiracy_analysis import BOT_SCORE_THRESHOLD
from conspiracy_analysis.analysis.statistics import compute_coadoption_matrix
from conspiracy_analysis.config import apply_conspiracy_config_to_graph, get_included_conspiracies
from conspiracy_analysis.utils.helpers import passes_bot_filter
from conspiracy_analysis.visualization import NHB_COL_DOUBLE, NHB_COL_ONE_HALF, NHB_PALETTE
from conspiracy_analysis.visualization import apply_panel_label, nhb_annotation_fontsize

G_coadoption, _ = load_graph_and_tweets(from_joblib=True, time_resolution='Hour')
print('Loaded G_MC.pkl')
nodes_to_remove = [
    node for node, data in G_coadoption.nodes(data=True)
    if not passes_bot_filter(G_coadoption, node, BOT_SCORE_THRESHOLD, 'HUMAN')
]
G_coadoption.remove_nodes_from(nodes_to_remove)
giant_nodes = max(nx.connected_components(G_coadoption.to_undirected()), key=len)
G_coadoption = G_coadoption.subgraph(giant_nodes).copy()
G_coadoption = apply_conspiracy_config_to_graph(G_coadoption)
coadoption_conspiracies = G_coadoption.graph.get('conspiracy_cols')
if coadoption_conspiracies is None:
    coadoption_conspiracies = get_included_conspiracies()

empirical_coadoption = compute_coadoption_matrix(G_coadoption, coadoption_conspiracies)

def compute_simulated_coadoption(scenario_results, conspiracies):
    n = len(conspiracies)
    matrices = []
    for run in scenario_results.runs:
        adopters = {c: set() for c in conspiracies}
        for node_id, hist in run.adoption_histories.items():
            for c in hist:
                if c in adopters:
                    adopters[c].add(node_id)
        mat = np.ones((n, n))
        for i in range(n):
            for j in range(i + 1, n):
                set_i = adopters[conspiracies[i]]
                set_j = adopters[conspiracies[j]]
                union = len(set_i | set_j)
                jaccard = len(set_i & set_j) / union if union > 0 else 0.0
                mat[i, j] = jaccard
                mat[j, i] = jaccard
        matrices.append(mat)
    mean_mat = np.mean(matrices, axis=0)
    return pd.DataFrame(mean_mat, index=conspiracies, columns=conspiracies)

simulated_coadoption = compute_simulated_coadoption(results_baseline, coadoption_conspiracies)

triu_idx = np.triu_indices(len(coadoption_conspiracies), k=1)
emp_values = empirical_coadoption.values[triu_idx]
sim_values = simulated_coadoption.values[triu_idx]
r = float(np.corrcoef(emp_values, sim_values)[0, 1])
ss_res = float(np.sum((emp_values - sim_values) ** 2))
ss_tot = float(np.sum((emp_values - emp_values.mean()) ** 2))
r_squared = 1.0 - ss_res / ss_tot if ss_tot > 0 else float('nan')
rmse = float(np.sqrt(np.mean((emp_values - sim_values) ** 2)))

short_labels = [display_conspiracy_label(c) for c in coadoption_conspiracies]
annot_fs = nhb_annotation_fontsize()

fig_heatmaps = plt.figure(figsize=(NHB_COL_DOUBLE[0] * 1.05, NHB_COL_DOUBLE[1] * 1.28))
heatmap_grid = fig_heatmaps.add_gridspec(
    1,
    3,
    width_ratios=[1, 1, 0.04],
    wspace=0.06,
)
ax_empirical = fig_heatmaps.add_subplot(heatmap_grid[0, 0])
ax_simulated = fig_heatmaps.add_subplot(heatmap_grid[0, 1])
cbar_ax = fig_heatmaps.add_subplot(heatmap_grid[0, 2])

sns.heatmap(
    empirical_coadoption.values,
    ax=ax_empirical,
    vmin=0,
    vmax=1,
    xticklabels=short_labels,
    yticklabels=short_labels,
    cmap='YlOrRd',
    annot=True,
    fmt='.2f',
    annot_kws={'size': annot_fs},
    cbar=False,
    square=True,
)
ax_empirical.set_title('Empirical Coadoption, Jaccard')
ax_empirical.tick_params(axis='x', labelsize=annot_fs, rotation=90, pad=2)
ax_empirical.tick_params(axis='y', labelsize=annot_fs, rotation=0)
for tick_label in ax_empirical.get_xticklabels():
    tick_label.set_ha('right')
    tick_label.set_va('center')
    tick_label.set_rotation_mode('anchor')
apply_panel_label(ax_empirical, 'a')

sns.heatmap(
    simulated_coadoption.values,
    ax=ax_simulated,
    vmin=0,
    vmax=1,
    xticklabels=short_labels,
    yticklabels=False,
    cmap='YlOrRd',
    annot=True,
    fmt='.2f',
    annot_kws={'size': annot_fs},
    cbar=True,
    cbar_ax=cbar_ax,
    square=True,
)
ax_simulated.set_title('Baseline Simulation Coadoption, Jaccard')
ax_simulated.tick_params(axis='x', labelsize=annot_fs, rotation=90, pad=2)
ax_simulated.tick_params(axis='y', left=False, labelleft=False)
ax_simulated.set_ylabel('')
for tick_label in ax_simulated.get_xticklabels():
    tick_label.set_ha('right')
    tick_label.set_va('center')
    tick_label.set_rotation_mode('anchor')
apply_panel_label(ax_simulated, 'b')

fig_heatmaps.subplots_adjust(bottom=0.38, top=0.88, left=0.08, right=0.94)
heatmap_save_path = os.path.join(FIGURE_FINAL_DIR, 'figA06_coadoption_jaccard.png')
fig_heatmaps.savefig(heatmap_save_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {heatmap_save_path}')

fig_scatter, ax_comparison = plt.subplots(figsize=NHB_COL_ONE_HALF)
ax_comparison.scatter(emp_values, sim_values, s=18, alpha=0.85,
                      color=NHB_PALETTE['positive'], linewidth=0)
lim = max(emp_values.max(), sim_values.max()) * 1.1
ax_comparison.plot([0, lim], [0, lim], '--', color='grey', alpha=0.5, linewidth=0.5)
ax_comparison.set_xlim(0, lim)
ax_comparison.set_ylim(0, lim)
ax_comparison.set_xlabel('Empirical Jaccard')
ax_comparison.set_ylabel('Simulated Jaccard')
ax_comparison.set_title('Pairwise Comparison')
ax_comparison.set_aspect('equal')
ax_comparison.text(
    0.05,
    0.95,
    f'Pearson r = {r:.3f}\nR squared = {r_squared:.3f}\nRMSE = {rmse:.3f}',
    transform=ax_comparison.transAxes,
    ha='left',
    va='top',
    fontsize=nhb_annotation_fontsize(),
    bbox={'boxstyle': 'round,pad=0.25', 'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.8},
)
print(f'r = {r:.3f}, R squared = {r_squared:.3f}, RMSE = {rmse:.3f}')
plt.tight_layout()
scatter_save_path = os.path.join(FIGURE_FINAL_DIR, 'figA06_coadoption_jaccard_scatter.png')
fig_scatter.savefig(scatter_save_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {scatter_save_path}')

## Appendix Figure 7

Exposure hazard ratio comparison across sequential adoption models.

In [ ]:
from conspiracy_analysis.visualization.figures import plot_exposure_hr_comparison

appendix_exposure_models = [
    'model_1', 'model_2a', 'model_3', 'model_4', 'model_5',
    'model_6', 'model_7', 'model_8',
]
missing_exposure_models = [name for name in appendix_exposure_models if name not in cox_results]
if missing_exposure_models:
    raise RuntimeError(
        f'Figure A07 requires fitted Cox models: {missing_exposure_models}.'
    )
appendix_exposure_terms = ['s7_d2', 's7_d3', 's7_d4']
appendix_exposure_boot = cox_bootstrap_results.get('exposure_intervals')
if not isinstance(appendix_exposure_boot, pd.DataFrame) or appendix_exposure_boot.empty:
    raise RuntimeError('Figure A07 requires bootstrap exposure intervals.')
for model_name in appendix_exposure_models:
    for term in appendix_exposure_terms:
        rows = appendix_exposure_boot[
            (appendix_exposure_boot['model'] == model_name)
            & (appendix_exposure_boot['term'] == term)
        ]
        if len(rows) != 1:
            raise RuntimeError(
                f'Figure A07 requires one bootstrap exposure interval for {model_name}, {term}.'
            )
        hr, lower, upper = rows[['hr', 'ci_lower', 'ci_upper']].iloc[0].astype(float)
        if not np.isfinite([hr, lower, upper]).all() or not lower <= hr <= upper:
            raise RuntimeError(
                f'Figure A07 has an invalid bootstrap exposure interval for {model_name}, {term}.'
            )

fig = plot_exposure_hr_comparison(
    cox_results,
    models_to_include=appendix_exposure_models,
    save_path=os.path.join(FIGURE_FINAL_DIR, 'figA07_exposure_hr_comparison.png'),
    bootstrap_intervals=cox_bootstrap_results,
)
plt.show()